In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

In [2]:
import numpy as np
import pandas as pd
from qiskit.quantum_info import Statevector
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score
from src.data_load_ionosphere import load_ionosphere
from src.preprocess import preprocess

In [3]:
features = 4                          # 4 features mean 2 qubits
seeds = [0, 1, 2, 3, 4]
C_grid = {"C": [0.1, 1, 10, 100, 1000]}
rbf_grid = {"C": [0.1, 1, 10, 100, 1000], "gamma": [0.001, 0.01, 0.1, 1, 10, "scale"]}


In [4]:
def normalize(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)

def to_states(X):
    return np.array([Statevector(row).data for row in X])

def metrics(y_true, pred):
    return (accuracy_score(y_true, pred),
            recall_score(y_true, pred, pos_label=0),
            f1_score(y_true, pred, average="macro"))


In [6]:
rows = []
for seed in seeds:
    X_train, X_test, y_train, y_test = load_ionosphere(seed)
    _, _, X_train_pca, X_test_pca, pca = preprocess(X_train, X_test, features)

    # positive shift first (avoids the sign-blind failure), then unit-length normalization
    mm = MinMaxScaler(feature_range=(0.1, 1))
    X_train_amp = normalize(mm.fit_transform(X_train_pca))
    X_test_amp = normalize(np.clip(mm.transform(X_test_pca), 0.1, 1))

    S_train, S_test = to_states(X_train_amp), to_states(X_test_amp)
    K_train = np.abs(S_train.conj() @ S_train.T) ** 2
    K_test = np.abs(S_test.conj() @ S_train.T) ** 2

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    q = GridSearchCV(SVC(kernel="precomputed"), C_grid, cv=cv, scoring="f1_macro").fit(K_train, y_train)
    acc, rec, f1 = metrics(y_test, q.predict(K_test))
    rows.append({"seed": seed, "model": "QSVM amplitude", "acc": acc, "recall_bad": rec, "f1": f1})

    r = GridSearchCV(SVC(kernel="rbf"), rbf_grid, cv=cv, scoring="f1_macro").fit(X_train_amp, y_train)
    acc, rec, f1 = metrics(y_test, r.predict(X_test_amp))
    rows.append({"seed": seed, "model": "RBF tuned (same input)", "acc": acc, "recall_bad": rec, "f1": f1})

    # control: the same kernel written classically
    p = GridSearchCV(SVC(kernel="poly", degree=2, gamma=1, coef0=0), C_grid, cv=cv, scoring="f1_macro").fit(X_train_amp, y_train)
    acc, rec, f1 = metrics(y_test, p.predict(X_test_amp))
    rows.append({"seed": seed, "model": "Poly deg-2 control", "acc": acc, "recall_bad": rec, "f1": f1})

    print(f"seed {seed} done", flush=True)
    pd.DataFrame(rows).to_csv("ionosphere_amplitude.csv", index=False)


seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done


In [7]:
df = pd.DataFrame(rows)
print(df.groupby("model")[["acc", "recall_bad", "f1"]].agg(["mean", "std"]).round(3).to_string())

                          acc        recall_bad            f1       
                         mean    std       mean    std   mean    std
model                                                               
Poly deg-2 control      0.828  0.039      0.568  0.131  0.787  0.058
QSVM amplitude          0.828  0.039      0.568  0.131  0.787  0.058
RBF tuned (same input)  0.856  0.045      0.752  0.131  0.837  0.057
